# MIQ - Frontend File

In [2]:
%pip install streamlit

  Using cached streamlit-1.56.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached pydeck-0.9.2-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.3-py3-none-any.whl.metadata (4.6 kB)
Using cached streamlit-1.56.0-py3-none-any.whl (9.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 kB 14.4 MB/s  0:00:00
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
Using cached gitdb-4.0.12-py3-none-any.whl (62 kB)
Using cached pydeck-0.9.2-py2.py3-none-any.whl (11.3 MB)
Using cached smmap-5.0.3-py3-none-any.whl (24 kB)
Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)
Using cached toml-0.10.2-py2.py3-none-any.whl (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 30.2 MB/s  0:00:01 30.6 MB/s eta 0:00

In [13]:
import streamlit as st
import pandas as pd
import requests
import json

## Configuration (Chris)

In [25]:
API_URL = "PLACEHOLDER"

# MEDIAN loaded from JSON file in directory
with open('feature_medians.json', 'r') as f:
    FEATURE_MEDIANS = json.load(f)

## User Inputation (Chris)

In [ ]:
def impute_features(user_input: dict, marathon_weather: str) -> dict:

    # Zero imputation
    user_input['previous_marathon_count'] = user_input.get('previous_marathon_count', 0)
    user_input['run_club_attendance_rate'] = user_input.get('run_club_attendance_rate', 0)

    # Median imputation
    for col, median in FEATURE_MEDIANS.items():
        if user_input.get(col, 0) == 0:
            user_input[col] = median

    # Weather OHE
    user_input['marathon_weather_Cold']  = 1 if marathon_weather == 'Cold'  else 0
    user_input['marathon_weather_Hot']   = 1 if marathon_weather == 'Hot'   else 0
    user_input['marathon_weather_Rainy'] = 1 if marathon_weather == 'Rainy' else 0
    user_input['marathon_weather_Windy'] = 1 if marathon_weather == 'Windy' else 0

    return user_input

## Page Setup (Corentin)

In [21]:
st.set_page_config(page_title="MarathonIQ", layout="wide")
st.title("MarathonIQ")
st.subheader("What actually predicts your marathon time?")

2026-04-23 16:39:52.538 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-23 16:39:52.539 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-23 16:39:52.539 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-23 16:39:52.540 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-23 16:39:52.540 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-23 16:39:52.541 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-23 16:39:52.541 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

## User Input / Output Fields (Corentin)

In [24]:
# Mock user input — test without Streamlit
mock_input = {
    'age': 38,
    'running_experience_months': 36,
    'weekly_mileage_km': 50,
    'injury_count': 0,
    'injury_severity': 0,
    'course_difficulty': 2,
    'vo2_max': 0,           # missing — should be imputed to 45.0
    'resting_heart_rate_bpm': 0,  # missing — should be imputed to 68.0
    'recovery_score': 0,    # missing — should be imputed to 6.0
    'nutrition_score': 0,   # missing — should be imputed to 5.1
    'previous_marathon_count': 0,
    'run_club_attendance_rate': 0,
}

result = impute_features(mock_input, marathon_weather='Neutral')
print(pd.Series(result))

age                          38.000000
running_experience_months    36.000000
weekly_mileage_km            50.000000
injury_count                  0.000000
injury_severity               0.000000
course_difficulty             2.000000
vo2_max                      53.632594
resting_heart_rate_bpm       78.000000
recovery_score                6.800000
nutrition_score               6.000000
previous_marathon_count       0.000000
run_club_attendance_rate      0.000000
marathon_weather_Cold         0.000000
marathon_weather_Hot          0.000000
marathon_weather_Rainy        0.000000
marathon_weather_Windy        0.000000
dtype: float64


In [7]:
# Confirm all 15 features present
EXPECTED_FEATURES = [
    'age', 'running_experience_months', 'weekly_mileage_km',
    'injury_count', 'injury_severity', 'course_difficulty',
    'vo2_max', 'resting_heart_rate_bpm', 'recovery_score',
    'nutrition_score', 'previous_marathon_count', 'run_club_attendance_rate',
    'marathon_weather_Cold', 'marathon_weather_Hot',
    'marathon_weather_Rainy', 'marathon_weather_Windy'
]

missing = [f for f in EXPECTED_FEATURES if f not in result]
extra   = [f for f in result if f not in EXPECTED_FEATURES]

print(f"Missing features: {missing}")
print(f"Extra features:   {extra}")
print(f"Complete: {len(missing) == 0}")

Missing features: []
Extra features:   []
Complete: True


In [8]:
# Mock prediction — replace with API call later
def mock_predict(feature_vector: dict) -> dict:
    prediction = 267  # placeholder
    hours   = prediction // 60
    minutes = prediction % 60
    return {
        'minutes': prediction,
        'formatted': f"{hours}h {minutes:02d}min"
    }

prediction = mock_predict(result)
print(f"Predicted finish time: {prediction['formatted']}")

Predicted finish time: 4h 27min
